# Initialize Pipeline Run

Creates the run record in `metadata.pipeline_runs` with configuration for all downstream tasks.

In [ ]:
import sys
import os

# Add src to path for imports
notebook_dir = os.getcwd()
repo_root = os.path.dirname(notebook_dir)
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import logging
logging.basicConfig(level=logging.INFO)

from verdict.orchestration import RunContext, load_config, get_environment
from verdict.orchestration.config_loader import get_default_config

In [ ]:
# Widget parameters
dbutils.widgets.text("catalog_name", "verdict_dev", "Unity Catalog Name")
dbutils.widgets.text("model_endpoint", "", "Model Endpoint (optional)")
dbutils.widgets.text("judge_endpoint", "", "Judge Endpoint (optional)")
dbutils.widgets.text("baseline_version", "", "Baseline Version (optional)")
dbutils.widgets.text("candidate_version", "", "Candidate Version (optional)")
dbutils.widgets.text("dataset_version", "", "Dataset Version (optional)")

catalog_name = dbutils.widgets.get("catalog_name")
overrides = {
    k: v for k, v in {
        "model_endpoint": dbutils.widgets.get("model_endpoint") or None,
        "judge_endpoint": dbutils.widgets.get("judge_endpoint") or None,
        "baseline_version": dbutils.widgets.get("baseline_version") or None,
        "candidate_version": dbutils.widgets.get("candidate_version") or None,
        "dataset_version": dbutils.widgets.get("dataset_version") or None,
    }.items() if v is not None
}

In [ ]:
# Load configuration
environment = get_environment()
logger = logging.getLogger(__name__)
logger.info(f"Environment: {environment}, Catalog: {catalog_name}")

try:
    config = load_config(environment=environment)
    logger.info(f"Loaded config from {environment}.yaml")
except Exception as e:
    logger.warning(f"Config file not found: {e}, using defaults")
    config = get_default_config()

# Apply catalog and overrides
config["catalog_name"] = catalog_name
config.update(overrides)

In [ ]:
# Get job context
job_id, job_run_id = None, None
try:
    context = dbutils.jobs.context()
    if context:
        job_id = context.get("jobId")
        job_run_id = context.get("runId")
        logger.info(f"Job context: job_id={job_id}, job_run_id={job_run_id}")
except Exception:
    logger.info("Not running in job context")

In [ ]:
# Initialize run record
ctx = RunContext(spark, catalog_name=catalog_name)
ctx.initialize_run(config=config, job_id=job_id, job_run_id=job_run_id)
ctx.start_run()
ctx.update_task_status("init_run", "completed")
logger.info(f"Initialized run: {ctx.run_id}")

In [ ]:
# Summary
print(f"\n{'='*60}")
print(f"RUN INITIALIZED: {ctx.run_id}")
print(f"{'='*60}")
print(f"Environment: {environment}")
print(f"Catalog: {catalog_name}")
print(f"\nConfiguration:")
for key, value in config.items():
    if not key.startswith("_"):
        print(f"  {key}: {value}")
print(f"{'='*60}\n")

In [ ]:
# Return values for downstream tasks
dbutils.jobs.taskValues.set("run_id", ctx.run_id)
dbutils.jobs.taskValues.set("catalog_name", catalog_name)